# Module C.1–C.3: Embeddings & Semantic Search
**Part II — Applied LLM Engineering**

> Represent meaning as vectors, search by similarity, discover structure in a corpus.

## 1. What Is an Embedding?

In Part I we saw **token embeddings**: a lookup table that maps each token ID to a dense vector. Those vectors are learned during pretraining and live inside the model.

A **sentence embedding** is different: it encodes the *meaning of an entire piece of text* into a single fixed-length vector. The model reads all the tokens, runs them through its layers, and pools the result into one vector per input string.

The key property: **semantically similar text lands close together in this vector space.** "Hello" and "Hi" should be near each other; "The sky is blue" should be further away.

We will use `all-MiniLM-L6-v2`, a 6-layer Transformer distilled for sentence similarity. It outputs 384-dimensional vectors and runs fast on CPU.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load the model once — subsequent calls use the cached weights
model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode three sentences; normalize_embeddings=True gives unit-norm vectors
sentences = ["Hello", "Hi", "The sky is blue"]
embs = model.encode(sentences, normalize_embeddings=True)

print("Shape:", embs.shape)          # (3, 384)
print("Dtype:", embs.dtype)
print("Norm of first vector:", np.linalg.norm(embs[0]).round(6))  # should be 1.0

## 2. Cosine Similarity from Scratch

Two vectors `a` and `b` are similar if they point in the same direction. **Cosine similarity** measures the angle between them:

$$\text{cos\_sim}(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$$

Range: **−1** (opposite) to **+1** (identical direction). For text embeddings we rarely see negatives.

When both vectors already have **unit norm** (which `normalize_embeddings=True` guarantees), `‖a‖ = ‖b‖ = 1`, so the formula collapses to:

$$\text{cos\_sim}(a, b) = a \cdot b$$

That is just a dot product — cheap and numerically stable.

In [ ]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

hello, hi, sky = embs[0], embs[1], embs[2]

sim_hello_hi  = cosine_sim(hello, hi)
sim_hello_sky = cosine_sim(hello, sky)

print(f'"Hello" vs "Hi"             : {sim_hello_hi:.4f}')
print(f'"Hello" vs "The sky is blue": {sim_hello_sky:.4f}')
print()
print("With normalized vectors, dot product == cosine sim:")
print(f'  dot(hello, hi)  = {float(np.dot(hello, hi)):.4f}  |  cosine = {sim_hello_hi:.4f}')

## 3. The Geometry of Meaning

Let's build a small corpus covering three topics — **programming**, **cooking**, and **sports** — and see whether same-topic sentences cluster together in embedding space.

We will visualize pairwise cosine similarities as a **heatmap**: bright cells = high similarity, dark cells = low similarity. No dimensionality reduction yet — just raw dot products.

In [ ]:
corpus_10 = [
    # programming (0-3)
    "Python is a popular high-level programming language.",
    "A for-loop iterates over items in a sequence.",
    "Functions are reusable blocks of code.",
    "Debugging means finding and fixing errors in code.",
    # cooking (4-6)
    "Sautéing vegetables in olive oil brings out their flavor.",
    "A sharp knife makes chopping onions much easier.",
    "Baking bread requires flour, water, yeast, and salt.",
    # sports (7-9)
    "Soccer is played with a round ball on a grass field.",
    "A marathon is a 26.2-mile running race.",
    "Basketball players score by shooting the ball through a hoop.",
]

embs_10 = model.encode(corpus_10, normalize_embeddings=True)
print("Corpus shape:", embs_10.shape)   # (10, 384)

# Pairwise similarity matrix: (10, 10)
# With normalized vectors this is just a matrix multiply
sim_matrix = embs_10 @ embs_10.T
print("Similarity matrix shape:", sim_matrix.shape)

In [ ]:
# Short labels for the heatmap axes
short_labels = [
    "Python", "for-loop", "functions", "debugging",   # programming
    "sautéing", "knife", "baking",                     # cooking
    "soccer", "marathon", "basketball",                 # sports
]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, vmin=0.0, vmax=1.0, cmap="Blues")

ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

# Annotate each cell with the score
for i in range(len(corpus_10)):
    for j in range(len(corpus_10)):
        val = sim_matrix[i, j]
        color = "white" if val > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

# Draw lines separating topic blocks
for boundary in [3.5, 6.5]:
    ax.axhline(boundary, color="red", linewidth=1.5, linestyle="--")
    ax.axvline(boundary, color="red", linewidth=1.5, linestyle="--")

fig.colorbar(im, ax=ax, label="Cosine similarity")
ax.set_title("Pairwise cosine similarity — 10-sentence corpus\n"
             "(red dashed lines separate topic blocks)", fontsize=11)
plt.tight_layout()
plt.show()

print("\nAverage within-topic similarities:")
blocks = [("programming", slice(0, 4)), ("cooking", slice(4, 7)), ("sports", slice(7, 10))]
for name, sl in blocks:
    block = sim_matrix[sl, sl]
    # exclude diagonal (self-similarity = 1.0)
    mask = ~np.eye(block.shape[0], dtype=bool)
    print(f"  {name}: {block[mask].mean():.4f}")

Notice the bright blocks along the diagonal — each topic cluster has higher internal similarity than cross-topic pairs. This is the geometry of meaning: **topic structure emerges automatically**, with no labels, just distances in 384-dimensional space.

## 4. Brute-Force Vector Index from Scratch

A vector index stores embeddings and, given a query vector, returns the most similar items. The simplest possible implementation: keep all vectors in a list, compute dot products against every stored vector at query time, and return the top-k.

This is **exact nearest-neighbor search**, and it scales linearly with the number of stored vectors — which is fine for small corpora.

In [ ]:
class VectorIndex:
    """Exact nearest-neighbor index backed by a plain NumPy matrix."""

    def __init__(self):
        self.vectors: list[np.ndarray] = []
        self.metadata: list = []

    def add(self, vector: np.ndarray, meta) -> None:
        """Add a single (normalized) vector with associated metadata."""
        self.vectors.append(vector)
        self.metadata.append(meta)

    def search(self, query_vec: np.ndarray, top_k: int = 3) -> list[tuple]:
        """
        Return the top_k most similar items as (metadata, score) tuples.
        Works because all stored vectors are normalized: dot product == cosine sim.
        """
        matrix = np.array(self.vectors)          # (n, d)
        scores = matrix @ query_vec              # (n,) dot products
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(self.metadata[i], float(scores[i])) for i in top_indices]

    def __len__(self):
        return len(self.vectors)


# Build index on the 10-sentence corpus
index = VectorIndex()
for sentence, vec in zip(corpus_10, embs_10):
    index.add(vec, sentence)

print(f"Index contains {len(index)} vectors of dim {embs_10.shape[1]}")

In [ ]:
def run_query(index: VectorIndex, query: str, top_k: int = 3) -> None:
    query_vec = model.encode([query], normalize_embeddings=True)[0]
    results = index.search(query_vec, top_k=top_k)
    print(f'Query: "{query}"')
    for rank, (text, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f}] {text}")
    print()

run_query(index, "how do I write a loop in Python?")
run_query(index, "what should I cook for dinner?")
run_query(index, "I want to watch a ball game")

## 5. Approximate Nearest Neighbors (ANN) — Concept Only

**The scaling problem.** Our `VectorIndex.search` is O(n) per query: it scores every stored vector. At 10 k vectors that is fine. At 1 M vectors with 384-dimensional floats, a single query touches ~1.5 GB of data. At 100 M vectors it becomes untenable.

**The solution: trade a small amount of accuracy for a huge speedup.** Two dominant approaches:

### HNSW (Hierarchical Navigable Small World)
Build a multi-layer graph where each node connects to its nearest neighbors. At query time, start from an entry point in the top (sparsest) layer and greedily navigate toward the query; each hop takes you closer. Descend to the next layer and repeat with finer granularity. Typical complexity: O(log n) hops.

### IVF (Inverted File Index)
Cluster the dataset into *k* centroids (e.g., k=1024). At query time, score only the *n-probe* nearest centroids and search within those clusters. You scan ~5–10% of vectors and miss very few true neighbors.

### Production libraries
| Library | Algorithm | Notes |
|---------|-----------|-------|
| **FAISS** (Meta) | IVF, HNSW, PQ | C++ core, best raw speed, GPU support |
| **Chroma** | HNSW (via hnswlib) | Python-native, great for RAG pipelines |
| **Weaviate / Qdrant / Pinecone** | HNSW variants | Managed vector databases |

We will not implement ANN here — that is a full module on its own. The key takeaway: **choose an ANN library for production; our `VectorIndex` is correct but linear**.

## 6. Semantic Search Pipeline

Let's wrap the encode + index + query flow into a clean class, then test it on a larger, mixed-topic corpus.

In [ ]:
class SemanticSearch:
    """End-to-end semantic search: encode corpus → index → query."""

    def __init__(self, corpus: list[str], model_name: str = "all-MiniLM-L6-v2"):
        self.corpus = corpus
        self.model = SentenceTransformer(model_name)
        print(f"Encoding {len(corpus)} documents …")
        embeddings = self.model.encode(corpus, normalize_embeddings=True, show_progress_bar=False)
        self._index = VectorIndex()
        for text, vec in zip(corpus, embeddings):
            self._index.add(vec, text)
        print("Index ready.")

    def search(self, query: str, top_k: int = 5) -> list[tuple[str, float]]:
        """Return (text, score) pairs for the top_k most relevant documents."""
        q_vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self._index.search(q_vec, top_k=top_k)

In [ ]:
# Larger mixed corpus — 27 Wikipedia-style sentences across 5 topics
large_corpus = [
    # Machine learning
    "Gradient descent minimizes a loss function by iteratively adjusting model weights.",
    "Overfitting occurs when a model memorizes training data and fails to generalize.",
    "A neural network with many hidden layers is called a deep neural network.",
    "Backpropagation computes gradients by applying the chain rule through layers.",
    "Cross-entropy is a common loss function for classification tasks.",
    # Natural language processing
    "Tokenization splits text into smaller units before feeding it to a language model.",
    "Attention allows a model to weigh the importance of different input tokens.",
    "BERT is pretrained on masked language modeling and next-sentence prediction.",
    "Named entity recognition identifies people, places, and organizations in text.",
    "Sentiment analysis classifies text as positive, negative, or neutral.",
    # Space exploration
    "The International Space Station orbits Earth at roughly 400 km altitude.",
    "Mars has two small moons called Phobos and Deimos.",
    "A rocket must reach escape velocity to leave Earth's gravitational pull.",
    "The Hubble Space Telescope has captured images of galaxies billions of light-years away.",
    "Astronauts on the ISS experience microgravity and must exercise daily to prevent muscle loss.",
    # Nutrition
    "Proteins are made of amino acids and are essential for building muscle tissue.",
    "Dietary fiber aids digestion and helps maintain healthy blood sugar levels.",
    "Vitamin D is synthesized by the skin when exposed to sunlight.",
    "Omega-3 fatty acids found in fish oil support heart and brain health.",
    "Excessive sodium intake is linked to higher blood pressure.",
    # History
    "The Roman Empire at its peak stretched from Britain to Mesopotamia.",
    "The printing press invented by Gutenberg revolutionized the spread of information.",
    "World War II ended in Europe on May 8, 1945, known as Victory in Europe Day.",
    "The Industrial Revolution began in Britain in the late 18th century.",
    "The Silk Road was an ancient trade network connecting China to the Mediterranean.",
    "The French Revolution began in 1789 and led to the rise of Napoleon Bonaparte.",
    "Ancient Egyptians built the Great Pyramid of Giza as a tomb for Pharaoh Khufu.",
]

ss = SemanticSearch(large_corpus)

In [ ]:
def show_results(query: str, results: list[tuple[str, float]]) -> None:
    print(f'Query: "{query}"')
    for rank, (text, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f}] {text}")
    print()

queries = [
    "how do neural networks learn from data?",
    "what happens in space beyond Earth?",
    "foods that keep you healthy",
    "important events in ancient history",
    "understanding the words in a sentence",
]

for q in queries:
    show_results(q, ss.search(q, top_k=3))

## 7. Clustering with K-Means from Scratch

Semantic search answers a specific query. **Clustering** asks a different question: *what structure is already in the corpus, with no query at all?*

K-means assigns each document to one of *k* clusters by iterating between two steps:
1. **Assignment**: each point goes to the nearest centroid.
2. **Update**: each centroid moves to the mean of its assigned points.

For cosine similarity on normalized vectors, "nearest centroid" means highest dot product. After updating, we re-normalize the centroid so the next iteration's dot products are valid cosine similarities.

In [ ]:
def kmeans(
    X: np.ndarray,
    k: int,
    n_iter: int = 20,
    random_state: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Spherical k-means on unit-norm vectors.
    Returns (labels, centroids) where labels[i] in {0, …, k-1}.
    """
    rng = np.random.default_rng(random_state)
    # Initialise centroids by sampling k data points at random
    idx = rng.choice(len(X), k, replace=False)
    centroids = X[idx].copy()   # (k, d), already unit-norm

    labels = np.zeros(len(X), dtype=int)
    for iteration in range(n_iter):
        # Assignment step: cosine sim == dot product for unit vectors
        sims = X @ centroids.T          # (n, k)
        new_labels = np.argmax(sims, axis=1)  # (n,)

        # Early stopping if assignments didn't change
        if np.all(new_labels == labels) and iteration > 0:
            print(f"  Converged at iteration {iteration}.")
            break
        labels = new_labels

        # Update step: compute mean and re-normalise
        for j in range(k):
            mask = labels == j
            if mask.any():
                mean_vec = X[mask].mean(axis=0)
                norm = np.linalg.norm(mean_vec)
                centroids[j] = mean_vec / norm if norm > 0 else centroids[j]

    return labels, centroids


# Encode the full large_corpus
embs_large = ss._index.vectors  # already computed by SemanticSearch
embs_large = np.array(embs_large)
print("Embedding matrix:", embs_large.shape)

# Run k-means with k=5 (we have 5 topics)
labels, centroids = kmeans(embs_large, k=5, n_iter=30, random_state=42)

In [ ]:
# Print which sentences landed in each cluster
for cluster_id in range(5):
    members = [large_corpus[i] for i in range(len(large_corpus)) if labels[i] == cluster_id]
    print(f"--- Cluster {cluster_id} ({len(members)} sentences) ---")
    for sent in members:
        print(f"  • {sent}")
    print()

Even with no labels, the clustering algorithm discovers topic groups from the geometry of the embedding space alone. Some clusters may mix history and NLP if the model places certain concepts close — that is a feature, not a bug: it reflects genuine semantic overlap.

## 8. 2-D Visualization with PCA (NumPy SVD)

384 dimensions cannot be directly plotted. **PCA** projects data into the directions of maximum variance. The trick: the principal components are the rows of `Vt` from the SVD of the data matrix.

We project each embedding onto the **first two principal components** to get 2-D coordinates, then scatter-plot them colored by cluster.

In [ ]:
# Center the data first (PCA operates on zero-mean data)
X_centered = embs_large - embs_large.mean(axis=0)

# SVD: U (n,n), S (min(n,d),), Vt (d,d)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Project onto the top-2 principal components
coords_2d = X_centered @ Vt[:2].T    # (n, 2)
print("2-D coordinates shape:", coords_2d.shape)

# Variance explained by the top 2 PCs
var_explained = (S[:2] ** 2) / (S ** 2).sum()
print(f"Variance explained: PC1={var_explained[0]:.1%}, PC2={var_explained[1]:.1%}")

In [ ]:
COLORS = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
TOPIC_NAMES = ["ML", "NLP", "Space", "Nutrition", "History"]  # true topics (for legend only)

fig, ax = plt.subplots(figsize=(11, 8))

for cluster_id in range(5):
    mask = labels == cluster_id
    ax.scatter(
        coords_2d[mask, 0],
        coords_2d[mask, 1],
        c=COLORS[cluster_id],
        s=80,
        alpha=0.85,
        label=f"Cluster {cluster_id}",
        zorder=3,
    )

# Add truncated sentence labels
for i, (text, xy) in enumerate(zip(large_corpus, coords_2d)):
    short = text[:40] + "…" if len(text) > 40 else text
    ax.annotate(
        short,
        xy=xy,
        xytext=(5, 3),
        textcoords="offset points",
        fontsize=6.5,
        alpha=0.8,
    )

ax.set_xlabel(f"PC 1 ({var_explained[0]:.1%} variance)", fontsize=11)
ax.set_ylabel(f"PC 2 ({var_explained[1]:.1%} variance)", fontsize=11)
ax.set_title("2-D PCA projection of sentence embeddings\nColored by k-means cluster (k=5)", fontsize=12)
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Try It Yourself

Three exercises to deepen your understanding.

In [ ]:
# Exercise (a)
# Build a semantic search over the course's first 26 notebook titles.
# Which notebooks come up when you query "how training works"?

notebook_titles = [
    "01 — Welcome & Setup",
    "02 — Prompt Anatomy",
    "03 — Temperature & Sampling",
    "04 — Tokenization Deep Dive",
    "05 — Token Embeddings",
    "06 — Attention Mechanism",
    "07 — Transformer Architecture",
    "08 — Pretraining Objectives",
    "09 — Fine-Tuning Basics",
    "10 — RLHF & Alignment",
    "11 — Prompt Engineering Patterns",
    "12 — Few-Shot & Chain-of-Thought",
    "13 — Tool Use & Function Calling",
    "14 — RAG — Retrieval-Augmented Generation",
    "15 — Agents & Planning",
    "16 — Evaluation Metrics",
    "17 — Safety & Red-Teaming",
    "18 — Hallucination & Grounding",
    "19 — Scaling Laws",
    "20 — Context Length & Long Documents",
    "21 — Multimodal Models",
    "22 — Structured Output",
    "23 — Latency & Throughput Optimization",
    "24 — Cost Management & Batching",
    "25 — Production Deployment Patterns",
    "26 — Observability & Monitoring",
]

# TODO: build a SemanticSearch over notebook_titles, then query it
# ss_titles = SemanticSearch(notebook_titles)
# show_results("how training works", ss_titles.search("how training works", top_k=5))

print("Exercise (a): build ss_titles and run the query above.")
print("Uncomment the two lines to run it.")

In [ ]:
# Exercise (b)
# Add a delete(index) method to VectorIndex.
# After deletion, searching should never return the deleted item.

class VectorIndexV2(VectorIndex):
    """VectorIndex extended with a delete method."""

    def delete(self, position: int) -> None:
        """Remove the item at the given position from the index."""
        # TODO: implement this
        # Hint: pop from both self.vectors and self.metadata
        raise NotImplementedError("Implement delete(position)")


# Test scaffold — uncomment once you implement delete()
# idx2 = VectorIndexV2()
# for sent, vec in zip(corpus_10, embs_10):
#     idx2.add(vec, sent)
# print("Before delete:", len(idx2), "items")
# idx2.delete(0)   # remove "Python is a popular ..."
# print("After delete:", len(idx2), "items")
# run_query(idx2, "Python programming")   # should NOT return the deleted sentence

print("Exercise (b): implement VectorIndexV2.delete() and run the test scaffold.")

In [ ]:
# Exercise (c)
# Try k=4 clusters on the large_corpus instead of k=5.
# Which topics get merged? Does the result make intuitive sense?

labels_4, _ = kmeans(embs_large, k=4, n_iter=30, random_state=42)

print("k=4 clustering results:")
for cluster_id in range(4):
    members = [large_corpus[i] for i in range(len(large_corpus)) if labels_4[i] == cluster_id]
    print(f"\n--- Cluster {cluster_id} ({len(members)} sentences) ---")
    for sent in members:
        print(f"  • {sent}")

## Summary

| Concept | Key idea | Where we used it |
|---------|----------|------------------|
| Sentence embedding | Encode meaning as a unit-norm vector | `model.encode(..., normalize_embeddings=True)` |
| Cosine similarity | Angle between vectors; = dot product when normalized | `cosine_sim`, `sim_matrix` heatmap |
| Exact search | Score every stored vector — O(n) per query | `VectorIndex` |
| ANN | Approximate, sublinear search via graphs/clusters | FAISS, Chroma |
| Semantic search | Encode query → score corpus → return top-k | `SemanticSearch` |
| Spherical k-means | Cluster normalized vectors; re-normalize centroids | `kmeans` |
| PCA (SVD) | Project to 2-D for visualization | `np.linalg.svd` |

**What's next** — Module C.4 will cover **retrieval-augmented generation (RAG)**: plugging a semantic index into an LLM so it can answer questions grounded in a private document corpus.